## Read the data

In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu

RUNS = 15
TOTAL_SPLITS = 1

# Read models

In [12]:
MODELS_DESCRIPTIONS = {
    "viterbi_simple_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 7,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 7,
        "core_end_state": 8

    },
    "viterbi_with_pseudo_and_weights_7": {
        "path": "experiment_results/core_finding_models_viterbi_simple_with_pseudo/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": ['HLA-DRB1*01:01',
                    # 'HLA-DPA1*01:03-DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": 35,
        "core_end_state": 36,
        "need_to_delete": [37, 38]

    },
    "viterbi_multiple_7": {
        "path": "experiment_results/core_multiple_allele/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 2,
        "alleles": ['HLA-DRB1*01:01',
                    'HLA-DPA1*01:03/DPB1*04:01'
                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [35, 36],
        "core_end_state": [37, 38]

    },
         "viterbi_models_multiple_motifs_21": {
        "path": "experiment_results/core_identification/",
        "states": 35,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [21, 22, 23],
        "core_end_state": [ 24,25, 26],
        "need_to_delete": [41, 42, 43, 44, 45, 46]

    },
     "viterbi_models_multiple_motifs_21_12_18": {
        "path": "experiment_results/core_identification/",
        "states": 21,
        "params_string": "30-min_iters",
        "combination_size": 1,
        "alleles": [
                 'HLA-DQA1*05:01/DQB1*02:01', 

                    # 'HLA-A*02:01'
                    ],
        "target_lengths": [12, 13, 14, 15, 16, 17, 18],
        "core_start_state": [21,22, 23],
        "core_end_state": [24,25,26],
        "need_to_delete": [27,28,29,30,31,32]

    },




    # "mix_of_alleles_balanced":
    # {
    #     "alleles": ['HLA-A*01:01', 'HLA-A*03:01', 'HLA-A*02:01'],
    #     "path": "experiment_results/alleles_with_groups_balanced/",
    #     "states": 25,
    #     "params_string": "5-min_iters",
    #     "target_lengths": [9, 10]
    # }
}

In [13]:
selected_models = [
     "viterbi_models_multiple_motifs_21_12_18"
]
models_to_read = ["v"]

In [14]:
from hmm_logic_methods import load_model

In [15]:
import itertools

name_to_run = dict()
name_to_split = dict()
name_to_type = dict()
name_to_model_root = dict()
name_to_target_lengths = dict()
selected_binding_models = list()
per_name_models = dict()


def populate_models_infos(model_name, run_num, split_num, model_path, model_type, target_lengths ):
    selected_binding_models.append(model_name)

    name_to_run[model_name] = run_num
    name_to_split[model_name] = split_num
    name_to_model_root[model_name] = model_path
    assert model_type in ["bw", "v", "bwv", "bw-it", "bwv-it", "v-it"] or "mixed" in model_type
    name_to_type[model_name] = model_type
    name_to_target_lengths[model_name] = target_lengths

for model in selected_models:
    model_description = MODELS_DESCRIPTIONS[model]
    combination_size = model_description['combination_size']
    model_alleles = model_description['alleles']
    MODELS_PATH = model_description['path']
    params_string = model_description['params_string']
    states = model_description['states']
    TARGET_LENGTHS = model_description['target_lengths']
    target_length_string = "-".join([str(l) for l in TARGET_LENGTHS])
    MIXED_ALLELES = list()

    for combination in itertools.combinations(sorted(model_alleles), combination_size):
        allele_name = "+".join(combination)
        # allele_name = f"{allele_name1}+{allele_name2}"
        MIXED_ALLELES.append(allele_name)

    for allele_name in MIXED_ALLELES:
        target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_').replace('/', '_')

        for run_num_b in range(RUNS):
            for split_num_b in range(TOTAL_SPLITS):
                #bw
                if 'bw' in models_to_read:
                    bw_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    print(bw_model_name)
                    populate_models_infos(bw_model_name, run_num_b,split_num_b, MODELS_PATH, f"bw-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'bwv' in models_to_read:
                    #bwv
                    bwv_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-baum-welch-alg-{params_string}_model-{target_allele_name}-{split_num_b}_plus_viterbi"
                    populate_models_infos(bwv_model_name, run_num_b,split_num_b, MODELS_PATH, f"bwv-it-mixed({target_allele_name})", TARGET_LENGTHS)
                if 'v' in models_to_read:
                    #v
                    v_model_name = f"{run_num_b}_run-{target_length_string}-{states}-st-viterbi-alg-{params_string}_model-{target_allele_name}-{split_num_b}"
                    populate_models_infos(v_model_name, run_num_b,split_num_b, MODELS_PATH, f"v-it-mixed({target_allele_name})", TARGET_LENGTHS)



for model_name in selected_binding_models:
    per_name_models[model_name] =  load_model(name_to_model_root[model_name], model_name)

In [16]:
per_name_models.keys()

dict_keys(['0_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '1_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '2_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '3_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '4_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '5_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '6_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '7_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '8_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '9_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0', '10_run-12-13-14-15-16-17-18-21-st-vite

## Reconstruct models

In [17]:
name_to_model_root

{'0_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0': 'experiment_results/core_identification/',
 '1_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0': 'experiment_results/core_identification/',
 '2_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0': 'experiment_results/core_identification/',
 '3_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0': 'experiment_results/core_identification/',
 '4_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0': 'experiment_results/core_identification/',
 '5_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0': 'experiment_results/core_identification/',
 '6_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0': 'experiment_results/core_identification/',
 '7_run-12-13-14-15-16-17-1

In [18]:
for model_name, model in per_name_models.items():
    states_to_delete = MODELS_DESCRIPTIONS[selected_models[0]]['need_to_delete']
    actual_states = [state for i, state in enumerate(model.states) if  i in states_to_delete]
    for actual_state in actual_states:
        model.graph.remove_node(actual_state)
    model.bake(verbose=1)

0_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0 : s024 summed to 0.00660066, normalized to 1.0
0_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0 : s025 summed to 0.0071653, normalized to 1.0
0_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0 : s026 summed to 0.00660066, normalized to 1.0
0_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0 : grouped_model-start summed to 0.00573952, normalized to 1.0
1_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0 : s024 summed to 0.00660066, normalized to 1.0
1_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0 : s025 summed to 0.0285602, normalized to 1.0
1_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0 : s026 summed to 0.00660066, normalized to 1.0
1_run-12-13-14

## Save models in same directories but in subdirectory of reconstructed

In [19]:
from hmm_visualization_methods import *

In [20]:
from hmm_logic_methods import save_model

per_name_target_path_to_results = dict()
for i, (model_name, model) in enumerate(per_name_models.items()):
    TARGET_PATH_TO_RESULTS = f'{name_to_model_root[model_name][:-1]}-reconstructed/'
    per_name_target_path_to_results[model_name] = TARGET_PATH_TO_RESULTS
    save_model(TARGET_PATH_TO_RESULTS, model,None)


In [21]:
%%time
import os

for i, model in enumerate(per_name_models.values()):
    TARGET_PATH_TO_RESULTS = per_name_target_path_to_results[model.name]
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
CPU times: total: 156 ms
Wall time: 3.65 s


In [22]:
%%time

for i, model in enumerate(per_name_models.values()):
    TARGET_PATH_TO_RESULTS = per_name_target_path_to_results[model.name]
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False, discrete=True, initial_params=None)
    print(i)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
CPU times: total: 50 s
Wall time: 50.8 s


In [23]:
from hmm_visualization_methods import *


per_name_pyviz = dict()
for name, model in per_name_models.items():
    TARGET_PATH_TO_RESULTS = per_name_target_path_to_results[model.name]
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0) (Delete zero edges)
Converting graph for model (1_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0) (Delete zero edges)
Converting graph for model (2_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0) (Delete zero edges)
Converting graph for model (3_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0) (Delete zero edges)
Converting graph for model (4_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0) (Delete zero edges)
Converting graph for model (5_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0) (Delete zero edges)
Converting graph for model (6_run-12-13-14-15-16-17-18-21-st-viterbi-alg-30-min_iters_model-HLA_DQA1_05_01_DQB1_02_01-0) (Delete zero edges)
Converting gr